# 🩺 MedStream — Data Cleaning Notebook

Welcome to the **MedStream** data cleaning notebook!

### What is this notebook about?
In this notebook, we will clean and prepare our new dataset: **`diabetes_prediction_dataset.csv`**.
This dataset has 100,000 patient records with information like age, gender, BMI, blood sugar levels, and whether the patient has diabetes.

### Our Simple Cleaning Steps:
1. **Load Data**: Read the CSV file into a pandas DataFrame.
2. **Explore Data**: Check column types, shape, and missing values.
3. **Remove Duplicate Rows**: Drop repeated patient records.
4. **Clean Gender**: Remove rare/invalid gender values (`Other`).
5. **Group Smoking History**: Simplify smoking categories so models understand them easily.
6. **Encode Categories to Numbers**: Convert words (like `Male`/`Female`) to simple numbers (0 and 1).
7. **Save Cleaned Data**: Export the clean dataset to `data/processed/diabetes_cleaned.csv`.

--- 
## Step 1: Import Libraries and Load Data
We use **pandas** to work with tables and **numpy** for numerical operations.

In [ ]:
import pandas as pd
import numpy as np

# File paths
raw_data_path = "data/raw/diabetes_prediction_dataset.csv"
processed_data_path = "data/processed/diabetes_cleaned.csv"

# Load the dataset
df = pd.read_csv(raw_data_path)

# Display the first 5 rows
print("Data loaded successfully!")
print("Dataset Shape:", df.shape)
df.head()

--- 
## Step 2: Basic Data Inspection
Let's check the data types and see if there are any missing (null) values.

In [ ]:
# Check information about columns and data types
print("=== Column Info ===")
df.info()

# Check for missing (NaN) values in each column
print("\n=== Missing Values Count ===")
print(df.isnull().sum())

--- 
## Step 3: Remove Duplicate Rows
Sometimes the same patient record is saved more than once. We will find and remove exact duplicates.

In [ ]:
# Count duplicates
duplicate_count = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_count:,}")

# Drop duplicates and keep the first occurrence
df = df.drop_duplicates()
print(f"Rows remaining after removing duplicates: {len(df):,}")

--- 
## Step 4: Clean the 'gender' Column
Let's see what values exist in the `gender` column. There are a few records with `Other` (only 18 records out of nearly 100,000). Removing them helps make our classification cleaner.

In [ ]:
# See gender counts
print("Gender counts before cleaning:")
print(df['gender'].value_counts())

# Keep only 'Female' and 'Male'
df = df[df['gender'].isin(['Female', 'Male'])]

print("\nGender counts after cleaning:")
print(df['gender'].value_counts())

--- 
## Step 5: Clean and Simplify 'smoking_history'
The `smoking_history` column currently has several categories:
- `never`
- `No Info`
- `current`
- `former`
- `not current`
- `ever`

To make it simple and clear, we can group them into 4 standard categories:
- **never**: People who never smoked.
- **current**: People who currently smoke.
- **former**: People who used to smoke (includes `former`, `not current`, `ever`).
- **No Info**: No smoking information available.

In [ ]:
# Group smoking history into 4 clear categories
def simplify_smoking(status):
    if status in ['former', 'not current', 'ever']:
        return 'former'
    elif status == 'current':
        return 'current'
    elif status == 'never':
        return 'never'
    else:
        return 'No Info'

df['smoking_history'] = df['smoking_history'].apply(simplify_smoking)

print("Simplified smoking history counts:")
print(df['smoking_history'].value_counts())

--- 
## Step 6: Convert Text Categories to Numbers (Encoding)
Machine learning models prefer numbers over text:
- `gender`: `Female` -> 0, `Male` -> 1
- `age`: Convert float ages (e.g. 80.0) to integers (80)

In [ ]:
# 1. Convert gender: Female -> 0, Male -> 1
df['gender_code'] = df['gender'].map({'Female': 0, 'Male': 1})

# 2. Convert age to integer
df['age'] = df['age'].astype(int)

# Show a sample of our transformed columns
df[['gender', 'gender_code', 'age', 'smoking_history', 'bmi', 'diabetes']].head()

--- 
## Step 7: Check the Target Variable ('diabetes')
Let's see how many patients have diabetes (`1`) versus no diabetes (`0`).

In [ ]:
target_counts = df['diabetes'].value_counts()
target_pct = df['diabetes'].value_counts(normalize=True) * 100

summary_table = pd.DataFrame({
    'Total Patients': target_counts,
    'Percentage (%)': target_pct.round(2)
})
summary_table.index = ['No Diabetes (0)', 'Has Diabetes (1)']
summary_table

--- 
## Step 8: Save the Cleaned Dataset
Now our data is clean, formatted, and ready for modeling! We will save it to `data/processed/diabetes_cleaned.csv`.

In [ ]:
import os

# Create data/processed folder if it does not exist
os.makedirs("data/processed", exist_ok=True)

# Save the cleaned data to CSV
df.to_csv(processed_data_path, index=False)

print("=" * 50)
print("SUCCESS: Cleaned dataset saved!")
print(f"File location: {processed_data_path}")
print(f"Total rows: {df.shape[0]:,}")
print(f"Total columns: {df.shape[1]}")
print("=" * 50)